# Covariate Data Preprocessing

This notebook builds the final **covariate file** for QTL analysis by combining known covariates, genotype principal components (PCs), and hidden factors.


## Description

Covariates correct for confounders so that detected QTLs reflect true genotype-phenotype associations rather than technical or population artifacts. This notebook assembles three kinds of covariates into one file:

1. **Known covariates** — measured variables from the fixed covariate file (e.g. `msex`, `age_death`, `pmi`, `study`).
2. **Genotype PCs** — the top PCs from the genotype PCA (`3_genotype_pca.ipynb`), which capture population structure.
3. **Hidden factors** — unobserved sources of variation (batch effects, technical noise) inferred from the phenotype residuals.

The two steps are: (1) **merge** the genotype PCs with the known covariates, and (2) **compute residuals** on the merged covariates and run **hidden-factor analysis** (PCA on the residuals, with the number of factors chosen by the Marchenko-Pastur distribution). The result is a single covariate file ready for TensorQTL.


## Input Files

| File | Description |
| --- | --- |
| `input/covariate/protocol_example.covariates.tsv` | Fixed/known covariates, one row per sample: `#id`, `msex`, `age_death`, `pmi`, `study`. |
| `output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.rds` | Genotype PCA model from `3_genotype_pca.ipynb`. |
| `output/genotype/genotype_pca/...prune.pca.scree.txt` | Scree table from `3_genotype_pca.ipynb`; used to pick how many genotype PCs to keep. |
| `output/rnaseq/protocol_example.rnaseq.bed.bed.gz` | Phenotype bed (from `1_phenotype_preprocessing.ipynb`); used to compute residuals for hidden-factor analysis. |


## Steps


### **Step 1.** Merge genotype PCs with known covariates

Merge the genotype PCA results with the known covariate file. `--k` sets how many genotype PCs to keep; here it is read from the scree table as the number of PCs that together explain < 80% of variance (15 for this dataset). `--tol-cov 0.4` is the maximum allowed missingness rate for covariates. The output is a merged covariate table (known covariates + genotype PCs).


In [ ]:
sos run pipeline/covariate_formatting.ipynb merge_genotype_pc \
    --cwd output/covariate \
    --pcaFile output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.rds \
    --covFile input/covariate/protocol_example.covariates.tsv \
    --tol-cov 0.4 \
    --k `awk '$3 < 0.8' output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.scree.txt | tail -1 | cut -f 1`


### **Step 2.** Compute residuals and run hidden-factor analysis

Using the phenotype and the merged covariates from Step 1, this step first regresses out the known covariates to obtain phenotype **residuals** (`Marchenko_PC_1`), then runs PCA on those residuals and keeps the components that pass the **Marchenko-Pastur** significance threshold as **hidden factors** (`Marchenko_PC_2`). `--mean-impute-missing` fills any missing covariate values with their column mean. The final output combines the known covariates, genotype PCs, and hidden factors — this is the covariate file used in downstream QTL analysis.


In [ ]:
sos run pipeline/covariate_hidden_factor.ipynb Marchenko_PC \
    --cwd output/covariate \
    --phenoFile output/rnaseq/protocol_example.rnaseq.bed.bed.gz \
    --covFile output/covariate/protocol_example.covariates.protocol_example.genotype.merged.plink_qc.protocol_example.genotype.king.unrelated.plink_qc.prune.pca.gz \
    --mean-impute-missing


## Output Files

| File | Description |
| --- | --- |
| `output/covariate/protocol_example.covariates.protocol_example.genotype...prune.pca.gz` | Merged covariates: known covariates + genotype PCs (output of Step 1). |
| `output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates...prune.pca.residual.bed.gz` | Phenotype residuals after regressing out known covariates (intermediate, Step 2). |
| `output/covariate/protocol_example.rnaseq.bed.protocol_example.covariates...prune.pca.Marchenko_PC.gz` | **Final covariate file**: known covariates + genotype PCs + hidden factors, ready for QTL analysis. |


## Anticipated Results

The final `*.Marchenko_PC.gz` file contains, as rows, the 4 known covariates (`msex`, `age_death`, `pmi`, `study`), 15 genotype PCs (`PC1`-`PC15`), and the hidden factors (`Hidden_Factor_PC1` onward) inferred by Marchenko-Pastur, with one column per sample. This single file is the covariate input for the downstream TensorQTL association analysis.
